In [8]:
import os
import re
import pandas as pd
from typing import List, Pattern, Tuple

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_YML_Files.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# --- helpers ---
def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# --- strip YAML comments (analyze only active lines) ---
YAML_LINE_COMMENT_RE = re.compile(r'(?m)^\s*#.*?$')
def strip_yaml_comments(raw: str) -> str:
    return YAML_LINE_COMMENT_RE.sub("", raw)

# === Define keyword SOURCES with explicit GROUPS ===
# Each entry: (group, label, [regexes...])

DEVICE_SOURCES: List[Tuple[str, str, List[str]]] = [
    # Real device via adb on-host
    ("Real_Device", "adb devices",       [r'(?m)^(?!\s*#)\s*adb\s+devices\b']),
    ("Real_Device", "adb get-state",     [r'(?m)^(?!\s*#)\s*adb\s+get-state\b']),
    ("Real_Device", "adb get-serialno",  [r'(?m)^(?!\s*#)\s*adb\s+get-serialno\b']),
    ("Real_Device", "adb install",       [r'(?m)^(?!\s*#)\s*adb\s+install(\s+-r)?\b']),
    ("Real_Device", "adb -s",            [r'(?m)^(?!\s*#)\s*adb\s+-s\s+\S+\b']),
    ("Real_Device", "adb shell",         [r'(?m)^(?!\s*#)\s*adb\s+shell\b']),
    ("Real_Device", "adb root",          [r'(?m)^(?!\s*#)\s*adb\s+root\b']),
    ("Real_Device", "adb settings",      [r'(?m)^(?!\s*#)\s*adb\s+shell\s+settings\b']),
    ("Real_Device", "adb input",         [r'(?m)^(?!\s*#)\s*adb\s+shell\s+input\b']),
    ("Real_Device", "adb pm grant",      [r'(?m)^(?!\s*#)\s*adb\s+shell\s+pm\s+grant\b']),

    # Emulator / managed virtual devices
    ("Emulator", "sdkmanager/avdmanager",[r'(?m)^(?!\s*#)\s*(sdkmanager|avdmanager)\b']),
    ("Emulator", "emulator -avd/@",      [r'(?m)^(?!\s*#)\s*emulator\s+(-avd|@)\S+']),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^(?!\s*#)\s*android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh",    [r'(?m)^(?!\s*#)\s*start-emulator\.sh\b']),
    ("Emulator", "reactivecircus runner",[r'uses:\s*reactivecircus/android-emulator-runner']),
    ("Emulator", "actions/setup-android",[r'uses:\s*actions/setup-android']),
    ("Emulator", "pierotofy/setup-android",[r'uses:\s*pierotofy/setup-android']),
    ("Emulator", "api-level",            [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ("Emulator", "abi/arch",             [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image",         [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name",          [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),

    # Third-party device labs
    ("Third_Party_Lab", "gcloud firebase",[r'(?m)^(?!\s*#)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",       [r'(?m)^(?!\s*#)\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack",[r'(?m)^(?!\s*#)\s*(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test android",[r'(?m)^(?!\s*#)\s*appcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",  [r'(?m)^(?!\s*#)\s*maestro\s+cloud\b']),
    ("Third_Party_Lab", "test_matrix/firebase.json",[r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]

TRIGGER_SOURCES: List[Tuple[str, str, List[str]]] = [
    # Gradle triggers
    ("Gradle", "connectedAndroidTest",   [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected(android)?test\b']),
    ("Gradle", "connected.*Android.*",   [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*android.*test\b']),
    ("Gradle", "connectedCheck",         [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connectedcheck\b']),
    ("Gradle", "createInstrCoverage",    [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*createinstrumentationtestcoveragereport\b']),
    ("Gradle", "runInstrumentationTests",[r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*runinstrumentationtests\b']),
    ("Gradle", "executeScreenshotTests", [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*executescreenshottests\b']),
    ("Gradle", "orchestrator task",      [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*orchestrator\b']),
    ("Gradle", "yaml script -> gradle",  [r'\bscript\s*:\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*']),
    ("Gradle", "connected (broad)",      [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*\b']),

    # ADB triggers
    ("ADB", "am instrument",             [r'(?m)^(?!\s*#)\s*(adb\s+shell\s+)?am\s+instrument\b']),

    # Third-party lab triggers
    ("Third_Party_Lab", "gcloud firebase",[r'(?m)^(?!\s*#)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",      [r'(?m)^(?!\s*#)\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run", [r'(?m)^(?!\s*#)\s*appcenter\s+test\s+run\s+android\b']),
]

# Precompile
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]
TRIGGER_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES]

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels = []
    groups = []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl)
            groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# === scan & export ===
rows = []

for fname in os.listdir(CONFIG_DIR):
    ext = os.path.splitext(fname)[1].lower()
    if ext not in ('.yml', '.yaml'):
        continue

    fpath = os.path.join(CONFIG_DIR, fname)
    if not os.path.isfile(fpath):
        continue

    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    # analyze only active lines
    content = strip_yaml_comments(raw).lower()

    device_labels, device_groups = collect_hits_with_groups(DEVICE_PATTERNS, content)
    trigger_labels, trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS, content)

    # parse full_name and ci_platform from: <full_name>__<platform>++<file>.yml
    full_name = "Unknown"
    ci_platform = "Unknown"
    base = os.path.basename(fname)
    if "__" in base and "++" in base:
        try:
            full_name = base.split("__", 1)[0]
            ci_platform = base.split("__", 1)[1].split("++", 1)[0]
        except Exception:
            pass

    rows.append({
        "filename": fname,
        "full_name": full_name,
        "ci_platform": ci_platform,
        "has_device_setup": bool(device_labels),
        "device_setup": ", ".join(device_labels),
        "device_setup_group": ", ".join(device_groups),
        "has_test_trigger": bool(trigger_labels),
        "test_trigger": ", ".join(trigger_labels),
        "test_trigger_group": ", ".join(trigger_groups),
    })

df = pd.DataFrame(rows, columns=[
    "filename", "full_name", "ci_platform",
    "has_device_setup", "device_setup", "device_setup_group",
    "has_test_trigger", "test_trigger", "test_trigger_group"
])
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV} (yaml files={len(df)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_YML_Files.csv (yaml files=12667)


looking at support files for CI isntro test with this logic:
- if Instru_test == True and instur_t_ci == False then
- look at the yaml files of those projects to search for a sign for call for a support file
- then read those support file for a potential instrumentation test

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import json
import glob
import pandas as pd
from pathlib import Path

try:
    import yaml  # pip install pyyaml
except ImportError:
    raise SystemExit("Please install pyyaml: pip install pyyaml")

# ========= CONFIG (edit if your paths differ) =========
MAIN_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.4_Total_Repo.csv"
ALL_CFG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
OUT_CSV = os.path.join(OUT_DIR, "3.4_CI_SupportingFile_Evidence.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# ========= DETECTION REGEX (case-insensitive) =========
CI_YAML_EXTS = {".yml", ".yaml"}

# Cloud labs (one-and-done proof)
RE_CLOUD = re.compile(
    r"(gcloud\s+firebase\s+test\s+android\s+run|saucectl\s+run|browserstack)",
    re.IGNORECASE,
)

# Triggers (Gradle / GMD / Flutter integration tests)
RE_TRIGGER = re.compile(
    r"(gradle(?:w|\.bat)?\s+.*\b(connectedAndroidTest|[\w:-]*managedDevice\w*AndroidTest)\b"
    r"|flutter\s+(?:test|drive)\b.*integration_test)",
    re.IGNORECASE,
)

# Device / execution setup (emulator provisioning)
RE_DEVICE = re.compile(
    r"(sdkmanager|avdmanager|\bemulator\b|android-emulator-runner|api[-_ ]?level)",
    re.IGNORECASE,
)

# Exclusions (unit-only or build-only)
RE_EXCLUDE = re.compile(
    r"gradle(?:w|\.bat)?\s+.*\b(build|check|test)\b|\bjvm[-_ ]?tests?\b",
    re.IGNORECASE,
)

# Local script / action / task calls
RE_LOCAL_SCRIPT = re.compile(
    r"(^|\s)(?P<cmd>(?:bash|sh)\s+)?(?P<path>\.?/?(?:ci|scripts|tools|\.github/actions)/[^\s;|&]+)",
    re.IGNORECASE,
)
RE_MAKE = re.compile(r"\bmake\s+(?P<target>[\w:-]+)", re.IGNORECASE)
RE_FASTLANE = re.compile(r"\bfastlane\s+(?P<lane>[\w:_-]+)", re.IGNORECASE)

# Composite/local action usage in YAML: uses: ./.github/actions/<name>
RE_LOCAL_ACTION_USES = re.compile(r"^\s*uses:\s*[\"']\s*\./\.github/actions/([^\"']+)", re.IGNORECASE)

# ========= HELPERS =========
def read_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return path.read_text(encoding="latin1", errors="ignore")
        except Exception:
            return ""

def list_repo_files(repo_id: str) -> list[Path]:
    """
    Heuristic: files may be saved as `<owner>.<repo>__Type++filename` anywhere under ALL_CFG_DIR,
    or in subfolders that include the repo_id in their path. We match both filename prefix and path contains.
    """
    results = []
    repo_id_lower = repo_id.lower()
    for p in Path(ALL_CFG_DIR).rglob("*"):
        if not p.is_file():
            continue
        name_lower = p.name.lower()
        path_lower = str(p).lower()
        if name_lower.startswith(repo_id_lower + "__") or repo_id_lower in path_lower:
            results.append(p)
    return results

def yaml_like(path: Path) -> bool:
    return path.suffix.lower() in CI_YAML_EXTS or path.name.lower() in (".travis.yml", "jenkinsfile", "azure-pipelines.yml", "bitrise.yml")

def parse_yaml_runs_uses(yaml_text: str):
    """
    Return (runs, uses_lines, raw_lines) from any YAML-ish config.
    runs: list of commands (strings) from steps[].run
    uses_lines: list of raw 'uses' strings (we handle composite actions separately)
    raw_lines: all lines (for regex scanning fallback)
    """
    lines = yaml_text.splitlines()
    runs = []
    uses_vals = []
    try:
        data = yaml.safe_load(yaml_text)
    except Exception:
        data = None

    if isinstance(data, dict):
        # GitHub Actions style
        jobs = data.get("jobs", {})
        if isinstance(jobs, dict):
            for job in jobs.values():
                steps = job.get("steps", [])
                if not isinstance(steps, list):
                    continue
                for st in steps:
                    if isinstance(st, dict):
                        if "run" in st and isinstance(st["run"], str):
                            runs.append(st["run"])
                        if "uses" in st and isinstance(st["uses"], str):
                            uses_vals.append(st["uses"])
        # Travis / GitLab fallback: try keys like 'script', 'before_script'
        for key in ("script", "before_script", "after_script"):
            if key in data:
                val = data[key]
                if isinstance(val, list):
                    runs.extend([str(x) for x in val])
                elif isinstance(val, str):
                    runs.append(val)
    else:
        # If we can't parse, we'll rely on regex over raw lines
        pass

    return runs, uses_vals, lines

def find_local_action_paths(uses_vals: list[str], raw_lines: list[str]) -> list[str]:
    local_actions = []
    # direct uses values with local path
    for u in uses_vals:
        if u.strip().startswith("./.github/actions/"):
            local_actions.append(u.strip())
    # also scan raw lines for "uses: ./.github/actions/..."
    for ln in raw_lines:
        m = RE_LOCAL_ACTION_USES.search(ln)
        if m:
            # normalize to a path
            local_actions.append("./.github/actions/" + m.group(1).strip())
    return list(dict.fromkeys(local_actions))  # dedupe

def grep_patterns(text: str, patterns: list[tuple[str, re.Pattern, str]]) -> list[dict]:
    hits = []
    for category, regex, confidence in patterns:
        for m in regex.finditer(text):
            # capture a short snippet
            s = m.start()
            line_start = text.rfind("\n", 0, s) + 1
            line_end = text.find("\n", m.end())
            if line_end == -1:
                line_end = len(text)
            snippet = text[line_start:line_end].strip()
            hits.append({"category": category, "snippet": snippet, "confidence": confidence})
    return hits

def match_supporting_files_from_runs(repo_files: list[Path], runs: list[str]) -> list[Path]:
    candidates = []
    # Collect possible file basenames from run commands
    basenames = set()
    for cmd in runs:
        # scripts
        for m in RE_LOCAL_SCRIPT.finditer(cmd):
            basenames.add(Path(m.group("path")).name)
        # make targets (we'll scan Makefile)
        if RE_MAKE.search(cmd):
            basenames.add("Makefile")
        # fastlane
        if RE_FASTLANE.search(cmd):
            basenames.add("Fastfile")
            basenames.add("Appfile")
    # Find repo files whose name ends with these basenames
    for p in repo_files:
        if p.name in basenames:
            candidates.append(p)
        # also include 'action.yml' in local action folders if present in repo files
        if p.name.lower() in {"action.yml", "action.yaml"}:
            candidates.append(p)
    # Always include shell scripts under typical dirs
    for p in repo_files:
        if any(seg in p.as_posix().lower() for seg in ("/ci/", "/scripts/", "/tools/")) and p.suffix.lower() in {".sh", ".bash", ".cmd", ".bat"}:
            candidates.append(p)
    # Deduplicate
    uniq = []
    seen = set()
    for p in candidates:
        if p not in seen:
            uniq.append(p)
            seen.add(p)
    return uniq

def analyze_yaml_and_support(repo_id: str, yaml_paths: list[Path], all_repo_files: list[Path]) -> list[dict]:
    """Return list of evidence dicts for this repo."""
    evidence = []
    for yml in yaml_paths:
        ytxt = read_text(yml)

        runs, uses_vals, raw_lines = parse_yaml_runs_uses(ytxt)

        # 1) Scan YAML itself for cloud/trigger/device
        hits_yaml = grep_patterns(
            ytxt,
            [
                ("cloud_lab", RE_CLOUD, "high"),
                ("trigger", RE_TRIGGER, "medium"),
                ("device_setup", RE_DEVICE, "medium"),
            ],
        )
        for h in hits_yaml:
            h.update({"repo": repo_id, "file": str(yml), "source": "yaml"})
        evidence.extend(hits_yaml)

        # 2) Follow local scripts/actions/Makefile/Fastlane
        local_actions = find_local_action_paths(uses_vals, raw_lines)
        support_files = match_supporting_files_from_runs(all_repo_files, runs)

        # If YAML referenced a local composite action folder, include its action.yml
        for act in local_actions:
            # act like "./.github/actions/x"
            act_base = Path(act).parts[-1].lower()
            for p in all_repo_files:
                if p.name.lower() in {"action.yml", "action.yaml"} and act_base in str(p.parent).lower():
                    support_files.append(p)

        # Analyze each supporting file
        for sup in support_files:
            stxt = read_text(sup)
            hits_sup = grep_patterns(
                stxt,
                [
                    ("cloud_lab", RE_CLOUD, "high"),
                    ("trigger", RE_TRIGGER, "medium"),
                    ("device_setup", RE_DEVICE, "medium"),
                ],
            )
            for h in hits_sup:
                h.update({"repo": repo_id, "file": str(sup), "source": "support"})
            evidence.extend(hits_sup)

    # Filter out pure exclusions if nothing positive matched
    if not any(ev["category"] in {"cloud_lab", "trigger", "device_setup"} for ev in evidence):
        # Still record a negative hint if we saw exclusions (optional)
        pass
    return evidence

# ========= MAIN =========
df = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
# normalize boolean-like
def truthy(x):
    return str(x).strip().lower() in {"true", "yes", "y", "1"}
def falsy(x):
    return str(x).strip().lower() in {"false", "no", "n", "0", ""}

# pick column names robustly
def pick_col(cands):
    cols_lower = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None

fullcol = pick_col(["full_name", "repo", "owner_repo"])
intru_col = pick_col(["Intru_test", "instru_test", "has_androidTest"])
ci_col = pick_col(["instru_t_ci", "ci_instrumentation", "ci_instru_detected"])

if not fullcol or not intru_col or not ci_col:
    raise ValueError("Could not locate required columns (full_name, Intru_test, instru_t_ci).")

# filter target repos: instrumentation present in sources, but CI not detected in YAML
targets = df[ df[intru_col].apply(truthy) & df[ci_col].apply(falsy) ].copy()
targets[fullcol] = targets[fullcol].astype(str).str.strip().str.lower()

rows = []
for repo_id in targets[fullcol].unique():
    # enumerate all files belonging to this repo
    repo_files = list_repo_files(repo_id)
    if not repo_files:
        rows.append({"repo": repo_id, "status": "no_files_found", "evidence_json": "[]"})
        continue

    # pick YAMLs among them
    yaml_paths = [p for p in repo_files if yaml_like(p)]
    if not yaml_paths:
        # If no YAML under All_Config_Files, nothing to follow
        rows.append({"repo": repo_id, "status": "no_yaml_found", "evidence_json": "[]"})
        continue

    # analyze YAML + supporting files
    ev = analyze_yaml_and_support(repo_id, yaml_paths, repo_files)

    if any(e["category"] == "cloud_lab" for e in ev):
        status = "exists_ci_high"
    elif any(e["category"] == "trigger" for e in ev) and any(e["category"] == "device_setup" for e in ev):
        status = "exists_ci_high"
    elif any(e["category"] in {"trigger", "device_setup"} for e in ev):
        status = "exists_ci_medium"
    else:
        status = "not_found"

    rows.append({
        "repo": repo_id,
        "status": status,
        "evidence_json": json.dumps(ev, ensure_ascii=False),
        "yaml_files_scanned": len(yaml_paths),
        "total_repo_files_scanned": len(repo_files),
    })

out_df = pd.DataFrame(rows).sort_values(["status", "repo"]).reset_index(drop=True)
out_df.to_csv(OUT_CSV, index=False)
print(f"Saved evidence summary -> {OUT_CSV}")
print(out_df["status"].value_counts(dropna=False))
